# Trabajo Práctico - Aprendizaje Automático

## Imports y Seed


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from six import StringIO  #pip3 install six

from IPython.display import Image, display
import pydotplus

import sklearn
from sklearn.model_selection import cross_validate, StratifiedKFold, ParameterGrid, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score, classification_report, confusion_matrix


In [ ]:
SEED = 2026

np.random.seed(SEED)
stratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## Separación de Datos de Desarrollo y Control?

In [ ]:
df_completo = pd.read_csv('data.csv')

In [ ]:
def train_test_split(df):
    buen_pronostico = df[df['target'] == 1]
    p = buen_pronostico.shape[0]/df.shape[0]

    mal_pronostico = df[df['target'] == 0]
    buenos = len(buen_pronostico)/len(df)
    filasBuenas = int(len(df)*buenos*p)
    filasMalas = int(len(df)*p-filasBuenas)
    # buscar como selecciona .sample()
    test = pd.DataFrame(columns=buen_pronostico.columns).astype(buen_pronostico.dtypes)
    test = pd.concat([test, buen_pronostico.sample(n=filasBuenas, random_state=SEED)], ignore_index=False)
    test = pd.concat([test, mal_pronostico.sample(n=filasMalas, random_state=SEED)], ignore_index=False)

    train = df.drop(test.index, axis=0)    
    return train.drop(columns=['target']), train['target'], test.drop(columns=['target']), test['target']

In [ ]:
# Definimos el dataset que usaremos para desarrollar el modelo y el dataset de control que usaremos para evaluar el modelo.
X_train, y_train, X_control_NOUSARTONTO, y_control_NOUSARTONTO = train_test_split(df_completo)

df_control_NOUSARTONTO = pd.concat([X_control_NOUSARTONTO, y_control_NOUSARTONTO], axis=1)
df = pd.concat([X_train, y_train], axis=1)

In [ ]:
# Legacy code
buen_pronostico = df[df['target'] == 1]
mal_pronostico = df[df['target'] == 0]
buenos = len(buen_pronostico)/500
# buscar como selecciona .sample()
control = pd.DataFrame(columns=buen_pronostico.columns)
control = pd.concat([control, buen_pronostico.sample(n=28)], ignore_index=False)
control = pd.concat([control, mal_pronostico.sample(n=72)], ignore_index=False)

# elegimos mantener las proporciones para no modificar la 
# distribución de los datos inicial, no elegimos random por que puede
# ocurrir que una clase quede subrepresentada y no se evalué correctamente
# el modelo sobre esa clase. 
# No hicimos nada más complejo para evitar data leakage.
control.index

## Ejer 2

### Ejer 2.1

In [ ]:
def train_tree(X_tr: np.ndarray, y_tr: np.ndarray, tree_params={}) -> DecisionTreeClassifier:
    arbol = DecisionTreeClassifier(**tree_params, random_state=SEED)
    arbol.fit(X_tr, y_tr)

    return arbol

def dibujar_arbol(clf, c_name, f_name):
    # modo de uso: dibujar_arbol(arbol, nombres_de_clases, nombres_de_features)
    if(c_name is None or f_name is None):
        return "Debe especificar los nombres de las clases y de las features"
    
    dot_data = StringIO()
    sklearn.tree.export_graphviz(clf, out_file = dot_data,
                    filled = True,
                    class_names = c_name,
                    feature_names = f_name,
                    special_characters = True)

    graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
    display(Image(graph.create_png()))

In [ ]:
X_train, y_train, X_test, y_test = train_test_split(df)

tree_1 = train_tree(X_train, y_train, tree_params={'max_depth': 3})

print("Score sobre el dataset de entrenamiento: ", tree_1.score(X_train, y_train))
print("Score sobre el dataset de prueba: ", tree_1.score(X_test, y_test))


In [ ]:
dibujar_arbol(tree_1, c_name=['0','1'], f_name=df.drop(columns=['target']).columns)

### Ejer 2.2

In [ ]:
# FALTA AÑADIR LAS AUCROC Y AUPRC
tree_2 = DecisionTreeClassifier(max_depth=3, random_state=SEED)

resultados_folds = []

y_global_true = np.empty(len(y_train))
y_global_pred = np.empty(len(y_train))
y_global_proba = np.empty(len(y_train)) # El nombre proba es por aucroc devuelve la probabilidad de pertenecer a la clase.

for fold_idx, (train_index, val_index) in enumerate(stratifiedKFold.split(X_train, y_train)):

    kf_X_train, kf_X_test = X_train.iloc[train_index], X_train.iloc[val_index]
    kf_y_train, kf_y_test = y_train.iloc[train_index], y_train.iloc[val_index]

    tree_2.fit(kf_X_train, kf_y_train)

    y_pred_train = tree_2.predict(kf_X_train)
    y_pred_proba_train = tree_2.predict_proba(kf_X_train)[:, 1] # Probabilidad de pertenecer a la clase positiva

    y_pred_test = tree_2.predict(kf_X_test)
    y_pred_proba_test = tree_2.predict_proba(kf_X_test)[:, 1] # Probabilidad de pertenecer a la clase positiva

    y_global_true[val_index] = kf_y_test
    y_global_pred[val_index] = y_pred_test
    y_global_proba[val_index] = y_pred_proba_test


    resultados_folds.append({
        'Permutación': str(fold_idx),
        'Accuracy (train)': accuracy_score(kf_y_train, y_pred_train),
        'Accuracy (validation)': accuracy_score(kf_y_test, y_pred_test),
        'AUC-PRC (train)': average_precision_score(kf_y_train, y_pred_proba_train), #segun gemini es average precision score es AUCPRC
        'AUC-PRC (validation)': average_precision_score(kf_y_test, y_pred_proba_test),
        'AUC-ROC (train)': roc_auc_score(kf_y_train, y_pred_proba_train),
        'AUC-ROC (validation)': roc_auc_score(kf_y_test, y_pred_proba_test)
    })

df_tabla = pd.DataFrame(resultados_folds)

promedios = df_tabla.drop(columns=['Permutación']).mean()
df_tabla.loc[len(df_tabla)] = ['Promedios'] + list(promedios)


df_tabla.loc[len(df_tabla)] = [
    'Global',
    '(NO)',
    accuracy_score(y_global_true, y_global_pred),
    '(NO)',
    average_precision_score(y_global_true, y_global_proba),
    '(NO)',
    roc_auc_score(y_global_true, y_global_proba)
]

In [ ]:
df_tabla

### Ejer 2.3

In [ ]:
def grid_search_tree(X_train, y_train, param_grid={'max_depth': [3,5,None], 'criterion': ['gini', 'entropy']}, metric='accuracy'):
    grid = ParameterGrid(param_grid)
    res = []
    
    for params in grid:
        current_tree = DecisionTreeClassifier(**params, random_state=SEED)
        
        cv_results = cross_validate(
            current_tree, 
            X_train, 
            y_train, 
            cv=stratifiedKFold, 
            scoring=metric, 
            return_train_score=True 
        )
        
        fila = {}
        for param_name, param_value in params.items():
            if param_name == 'max_depth' and param_value is None:
                fila[param_name] = 'Infinito'
            else:
                fila[param_name] = param_value
                
        fila[f'{metric} (training)'] = cv_results['train_score'].mean()
        fila[f'{metric} (validación)'] = cv_results['test_score'].mean()
        
        res.append(fila)

    return pd.DataFrame(res)

### DUDA: Nos dice que usemos promedios pero eso no estaba mal? ScikitLearn en sus funciones hace promedio para cualquier score, incluso metricas no lineales

In [ ]:
res = grid_search_tree(X_train, y_train)

In [ ]:
res

## Ejer 3

#### Ejemplo

In [ ]:
# Al usar RandomizedSearchCV, ya realiza todo lo que hicimos antes, ejemplo con tree:
random_search_tree = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=SEED),
    param_distributions={
        'max_depth': [3, 5, 6, 7, None],
        'criterion': ['gini', 'entropy']
    }, # restringe el espacio de búsqueda a estos valores (prueba combinaciones dentro de los rangos de estos valores, en este caso son discretos así que solo prueba combinaciones de estos)
    n_iter=10, # combinaciones aleatorias a probar
    scoring='accuracy',
    cv=stratifiedKFold,
    random_state=SEED,
    return_train_score=True,
    n_jobs=-1 # esto usa todos los nucleos de CPU, stonks
)

random_search_tree.fit(X_train, y_train)

### Ejer 3.2

#### Naive Bayes

In [ ]:
nb_model = GaussianNB()

resultados_nb = cross_validate(
    nb_model, 
    X_train,
    y_train, 
    cv=stratifiedKFold, 
    scoring='roc_auc'
)

aucroc_promedio_nb = resultados_nb['test_score'].mean()
print(f"AUCROC promedio de Naïve Bayes (Default): {aucroc_promedio_nb:.4f}")

## Ejer 4

## Viendo cosas de varianza

In [ ]:
std_total = df.drop(columns=['target']).std()
media_total = df.drop(columns=['target']).mean()
cv_total = (std_total / media_total).abs()

var_total = df.drop(columns=['target']).var()
var_por_clase = df.groupby('target').var().T

df_stats = pd.DataFrame({
    'Varianza_Total': var_total,
    'CV_Total': cv_total,
    'Varianza_Clase_0': var_por_clase[0], # Ni idea si sirve de algo para la clase, pero ya que estoy
    'Varianza_Clase_1': var_por_clase[1]
})

In [ ]:
print(f"Genes con CV < 10: {df_stats[df_stats['CV_Total'] < 10].shape[0]} de {df_stats.shape[0]}")
display(df_stats.sort_values(by='CV_Total', ascending=True))
# Nota: No hay genes tales que la varianza sea casi 0, todas son considerables pero hay muchas excepcionalmente altas. Sería interesante ver porqué

### Correlacion

In [ ]:
X = df.drop(columns=['target'])

matriz_corr = X.corr(method='spearman')

mask = np.triu(np.ones_like(matriz_corr, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(
    matriz_corr, 
    mask=mask, 
    cmap='coolwarm', 
    center=0, 
    vmin=-1, 
    vmax=1, 
    cbar_kws={'label': 'Coeficiente de Correlación (Spearman)'}
)
plt.title('Matriz de Correlación de Expresión Genética (ARN)')
plt.xlabel('Genes')
plt.ylabel('Genes')
plt.tight_layout()
plt.show()

correlaciones_apiladas = matriz_corr.mask(mask).stack()
correlaciones_fuertes = correlaciones_apiladas[abs(correlaciones_apiladas) > 0.5].sort_values(ascending=False)
print(correlaciones_fuertes) # Qué gen tiene una correlación marcada.

## Normalidad de Atributos


In [ ]:
from scipy.stats import shapiro, normaltest, anderson

genes_df = df.drop(columns=['target'])
resultados = []

for gen in genes_df.columns:
    data = genes_df[gen].dropna()
    
    _, p_shapiro = shapiro(data)
    es_normal_shapiro = p_shapiro > 0.05
    
    _, p_dagostino = normaltest(data)
    es_normal_dagostino = p_dagostino > 0.05
    
    res_anderson = anderson(data, dist='norm')
    es_normal_anderson = res_anderson.statistic < res_anderson.critical_values[2]
    
    resultados.append({
        'Gen': gen,
        'p_valor_Shapiro': p_shapiro,
        'Normal_Shapiro': es_normal_shapiro,
        'p_valor_DAgostino': p_dagostino,
        'Normal_DAgostino': es_normal_dagostino,
        'Estadistico_Anderson': res_anderson.statistic,
        'Normal_Anderson': es_normal_anderson
    })

df_tests_normalidad = pd.DataFrame(resultados)
display(df_tests_normalidad.sort_values(by='p_valor_Shapiro', ascending=True).head(15))

# print contar cuantos genes pasaron cada test
print("Genes que pasaron el test de Shapiro-Wilk:", df_tests_normalidad['Normal_Shapiro'].sum())
print("Genes que pasaron el test de D'Agostino:", df_tests_normalidad['Normal_DAgostino'].sum())
print("Genes que pasaron el test de Anderson-Darling:", df_tests_normalidad['Normal_Anderson'].sum())

In [ ]:
resultados_condicionales = []
clases = df['target'].unique()

for clase in clases:
    df_clase = df[df['target'] == clase].drop(columns=['target'])
    
    for gen in df_clase.columns:
        data = df_clase[gen].dropna()
        
        _, p_shapiro = shapiro(data)
        es_normal_shapiro = p_shapiro > 0.05
        
        _, p_dagostino = normaltest(data)
        es_normal_dagostino = p_dagostino > 0.05
        
        # Test de Anderson-Darling
        res_anderson = anderson(data, dist='norm')
        es_normal_anderson = res_anderson.statistic < res_anderson.critical_values[2]
        
        resultados_condicionales.append({
            'Clase': clase,
            'Gen': gen,
            'p_valor_Shapiro': p_shapiro,
            'Normal_Shapiro': es_normal_shapiro,
            'p_valor_DAgostino': p_dagostino,
            'Normal_DAgostino': es_normal_dagostino,
            'Estadistico_Anderson': res_anderson.statistic,
            'Normal_Anderson': es_normal_anderson
        })

df_tests_condicional = pd.DataFrame(resultados_condicionales)

display(df_tests_condicional.sort_values(by=['Clase', 'p_valor_Shapiro'], ascending=[True, True]).head(15))

for clase in sorted(clases):
    df_sub = df_tests_condicional[df_tests_condicional['Clase'] == clase]
    print(f"\n--- Conteo para la Clase {clase} ---")
    print("Genes que pasaron Shapiro-Wilk:", df_sub['Normal_Shapiro'].sum())
    print("Genes que pasaron D'Agostino:", df_sub['Normal_DAgostino'].sum())
    print("Genes que pasaron Anderson-Darling:", df_sub['Normal_Anderson'].sum())

#### Histograma de cada atributo en su totalidad

In [ ]:
import matplotlib.pyplot as plt
import math

genes_df = df.drop(columns=['target'])

n_cols = 4
n_rows = math.ceil(genes_df.shape[1] / n_cols)

alto_figura = n_rows * 3

genes_df.hist(
    bins=30,
    layout=(n_rows, n_cols),
    figsize=(20, alto_figura),
    edgecolor='black',
    grid=False
)

plt.tight_layout()
plt.show()

## Distribucion por clase

In [ ]:
import pingouin as pg
from IPython.display import display
import warnings

def tests_multivariables(df, nombre_dataset="Dataset", target_col='target'):
    warnings.filterwarnings('ignore')
    
    print(f"RESULTADOS PARA: {nombre_dataset}")
    print("-" * 50)
    
    X_clase_0 = df[df[target_col] == 0].drop(columns=[target_col])
    X_clase_1 = df[df[target_col] == 1].drop(columns=[target_col])
    
    mvn_0 = pg.multivariate_normality(X_clase_0)
    mvn_1 = pg.multivariate_normality(X_clase_1)
    
    print("Test de Normalidad Multivariable (Henze-Zirkler):")
    print(f"Clase 0 -> Normal: {mvn_0.normal} | p-valor: {mvn_0.pval}")
    print(f"Clase 1 -> Normal: {mvn_1.normal} | p-valor: {mvn_1.pval}\n")
    
    lista_genes = df.drop(columns=[target_col]).columns.tolist()
    
    try:
        res_box = pg.box_m(df, dvs=lista_genes, group=target_col)
        print("Test de Igualdad de Covarianzas (Box's M):")
        print(f"Matrices iguales: {res_box['equal_cov'].iloc[0]} | p-valor: {res_box['pval'].iloc[0]}")
        display(res_box)
    except Exception as e:
        print("Test de Igualdad de Covarianzas (Box's M):")
        print(f"ERROR MATEMÁTICO: {e}")
        
    print("=" * 60, "\n")

In [ ]:
tests_multivariables(df, nombre_dataset="Dataset de Entrenamiento", target_col='target')

genes_df = df.drop(columns=['target'])
genes_normales = []

for gen in genes_df.columns:
    data = genes_df[gen].dropna()
    _, p_valor = shapiro(data)
    
    if p_valor > 0.05:
        genes_normales.append(gen)

columnas_finales = genes_normales + ['target']
df_shapiro = df[columnas_finales].copy()

tests_multivariables(df_shapiro, nombre_dataset="Dataset de Entrenamiento (Shapiro)", target_col='target')

In [33]:
target = 'target'
features = [col for col in df.columns if col != target]

for feature in features:
    
    plt.figure(figsize=(8, 5))
    
    sns.histplot(
        data=df,
        x=feature,
        hue=target,
        kde=True,
        stat="density",
        common_norm=False,
        bins=30,
        alpha=0.4
    )
    
    plt.title(f"Distribución de {feature} por clase")
    plt.xlabel(feature)
    plt.ylabel("Densidad")
    
    #plt.show()

KeyboardInterrupt: 

In [48]:
from sklearn.neighbors import KNeighborsClassifier



# Al usar RandomizedSearchCV, ya realiza todo lo que hicimos antes, ejemplo con tree:
rsk = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions={
        'n_neighbors': range(1,20),
        'weights': ["uniform","distance"],
        'metric':["cityblock","cosine","euclidean"]
    }, 
    n_iter=75, 
    scoring='roc_auc',
    cv=stratifiedKFold,
    random_state=SEED,
    return_train_score=True,
    
)

rsk.fit(X_train, y_train)



,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'metric': ['cityblock', 'cosine', ...], 'n_neighbors': range(1, 20), 'weights': ['uniform', 'distance']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",75
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",2026
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for 

In [49]:
print(rsk.score(X_train,y_train))
print(rsk.score(X_test,y_test))

1.0
0.8045366795366795


https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE111113
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE103275
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE106273

https://bibliotecadigital.exactas.uba.ar/download/tesis/tesis_n7384_GarciaSola.pdf